# ECS vs Jaccard AUC Validation on Wikipedia Pairs

This notebook validates the **Edit Clustering Score (ECS)** hypothesis for near-duplicate text detection.

**Hypothesis**: ECS — defined as the Index of Dispersion (IoD) of inter-edit-gap positions from a word-level LCS diff — should be higher for near-duplicate pairs than for hard-negative (same-topic, different article) pairs. If so, ECS should improve AUC on top of Jaccard 5-gram similarity.

**Experiment**:
- Text pairs are categorized as `near_dup` (one article spliced into another), `hard_neg` (same topic prefix, different articles), or `random` (cross-topic).
- Features computed: Jaccard 5-gram similarity and ECS (IoD of edit gap positions).
- Results evaluate whether ECS adds value over Jaccard alone.

**Key finding (from full run)**: DISCONFIRMED — near-duplicates actually have *lower* IoD than hard-negatives because splicing creates ONE large contiguous edit block, while hard-negatives have many scattered differences.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import sys
import json
import random
import difflib
import math
import gc
from collections import defaultdict

import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-f2202a-edit-clustering-score-spatial-edit-patte/main/round-1/evaluation-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
pairs = data['pairs']
print(f"Loaded {len(pairs)} pairs")
print(f"  near_dup: {sum(1 for p in pairs if p['pair_type']=='near_dup')}")
print(f"  hard_neg: {sum(1 for p in pairs if p['pair_type']=='hard_neg')}")
print(f"  random:   {sum(1 for p in pairs if p['pair_type']=='random')}")
print(f"\nSample pair (near_dup):")
nd = next(p for p in pairs if p['pair_type']=='near_dup')
print(f"  text1[:80]: {nd['text1'][:80]}")
print(f"  text2[:80]: {nd['text2'][:80]}")

## Config

Tunable parameters for the demo. The original full run used `PAIRS_PER_CLASS=300`, `N_ARTICLES=3000`, and `BOOTSTRAP_B=2000`. Here we use the pre-loaded mini dataset (no Wikipedia streaming needed) and reduce bootstrap iterations for speed.

In [ ]:
# --- Config (set to minimum for fast demo) ---
RANDOM_SEED = 42
BOOTSTRAP_B = 200     # original: 2000
# PAIRS_PER_CLASS and N_ARTICLES are not needed — data is pre-loaded from mini_demo_data.json

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Feature Functions

Two features are computed for each text pair:
1. **Jaccard 5-gram similarity**: fraction of shared 5-word shingles between the two texts.
2. **ECS (Edit Clustering Score)**: Index of Dispersion of the positions of edit operations found by a word-level LCS diff. High IoD means edits are spread out; low IoD means they are clustered into one block.

In [ ]:
def jaccard_5gram(t1: str, t2: str) -> float:
    def shingles(text: str, k: int = 5):
        words = text.lower().split()
        return set(tuple(words[i:i+k]) for i in range(len(words)-k+1))
    s1, s2 = shingles(t1), shingles(t2)
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)


def compute_ecs(t1: str, t2: str) -> dict:
    """Compute ECS (IoD of inter-edit-gap lengths) and auxiliary features."""
    w1 = t1.lower().split()
    w2 = t2.lower().split()
    total_len = len(w1)
    matcher = difflib.SequenceMatcher(None, w1, w2, autojunk=False)
    opcodes = matcher.get_opcodes()

    edit_positions = []
    run = 0
    max_run = 0
    for tag, i1, i2, j1, j2 in opcodes:
        if tag != 'equal':
            mid = (i1 + i2) / 2.0
            edit_positions.append(mid)
            run += (i2 - i1)
            max_run = max(max_run, run)
        else:
            run = 0

    n_edits = len(edit_positions)
    edit_count_norm = n_edits / max(total_len, 1)
    longest_run = max_run / max(total_len, 1)
    edit_span_frac = 0.0

    if n_edits >= 2:
        edit_span_frac = (edit_positions[-1] - edit_positions[0]) / max(total_len, 1)
        gaps = np.diff(edit_positions)
        mean_gap = float(np.mean(gaps))
        iod = float(np.var(gaps) / mean_gap) if mean_gap > 0 else 0.0
    else:
        iod = 0.0

    return {
        'ecs': iod,
        'edit_count': n_edits,
        'edit_count_norm': edit_count_norm,
        'edit_span_frac': edit_span_frac,
        'longest_run': longest_run,
    }

## Compute Features

Compute Jaccard and ECS for every pair in the loaded dataset.

In [ ]:
results = []
for i, p in enumerate(pairs):
    jac = jaccard_5gram(p['text1'], p['text2'])
    ecs_feats = compute_ecs(p['text1'], p['text2'])
    results.append({
        **p,
        'jaccard': jac,
        **ecs_feats,
    })

print(f"Features computed for {len(results)} pairs")
print(f"\nSample near_dup result:")
r = next(r for r in results if r['pair_type']=='near_dup')
print(f"  jaccard={r['jaccard']:.4f}  ecs={r['ecs']:.4f}  edit_count={r['edit_count']}")
print(f"\nSample hard_neg result:")
r = next(r for r in results if r['pair_type']=='hard_neg')
print(f"  jaccard={r['jaccard']:.4f}  ecs={r['ecs']:.4f}  edit_count={r['edit_count']}")

## Evaluation Metrics

Three evaluation criteria from the hypothesis:
1. **AUC improvement**: Does Jaccard+ECS beat Jaccard alone on the hard-negative subset?
2. **ECS alone**: Is ECS-only AUC > 0.65?
3. **IoD ratio**: Is median IoD higher for near-duplicates than hard-negatives (ratio ≥ 2.0)?

Helper functions below are copied verbatim from the original `eval.py`.

In [ ]:
def bootstrap_auc_delta(y_true, score1, score2, B: int = 2000, seed: int = 0) -> tuple:
    """Bootstrap CI for AUC(score2) - AUC(score1)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        d = roc_auc_score(yt, score2[idx]) - roc_auc_score(yt, score1[idx])
        deltas.append(d)
    deltas = np.array(deltas)
    return float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))


def confusion_at_recall(y_true, scores, target_recall: float = 0.8) -> dict:
    """Find threshold achieving >= target_recall; report confusion matrix."""
    thresholds = np.sort(np.unique(scores))[::-1]
    best = None
    for thr in thresholds:
        pred = (scores >= thr).astype(int)
        tp = int(np.sum((pred == 1) & (y_true == 1)))
        fn = int(np.sum((pred == 0) & (y_true == 1)))
        fp = int(np.sum((pred == 1) & (y_true == 0)))
        tn = int(np.sum((pred == 0) & (y_true == 0)))
        recall = tp / max(tp + fn, 1)
        if recall >= target_recall:
            prec = tp / max(tp + fp, 1)
            best = {'threshold': float(thr), 'precision': prec, 'recall': recall,
                    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}
            break
    if best is None:
        best = {'threshold': float(thresholds[-1]), 'precision': 0.0, 'recall': 0.0,
                'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
    return best


def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    pooled_std = math.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return float((np.mean(a) - np.mean(b)) / pooled_std) if pooled_std > 0 else 0.0


def length_bucket(avg_words: float) -> str:
    if avg_words < 200:
        return '<200'
    elif avg_words <= 500:
        return '200-500'
    else:
        return '>500'

## Statistical Analysis

Run the full evaluation on the hard-negative subset (near_dup vs hard_neg only, excluding random pairs).

In [ ]:
# ── Hard-negative subset (near-dup vs hard-neg only) ──
hard_subset = [d for d in results if d['pair_type'] in ('near_dup', 'hard_neg')]
print(f"Hard-neg subset: {len(hard_subset)} pairs")

y_hard = np.array([d['label'] for d in hard_subset])
jac_hard = np.array([d['jaccard'] for d in hard_subset])
ecs_hard = np.array([d['ecs'] for d in hard_subset])
combined_hard = jac_hard + 0.3 * ecs_hard  # simple linear combination

auc_jac = float(roc_auc_score(y_hard, jac_hard))
auc_combined = float(roc_auc_score(y_hard, combined_hard))
auc_ecs_only = float(roc_auc_score(y_hard, ecs_hard))
delta_auc = auc_combined - auc_jac
print(f"AUC Jaccard={auc_jac:.4f}  Combined={auc_combined:.4f}  ECS-only={auc_ecs_only:.4f}  delta={delta_auc:.4f}")

ci_low, ci_high = bootstrap_auc_delta(y_hard, jac_hard, combined_hard, B=BOOTSTRAP_B)
print(f"Bootstrap 95% CI for delta_AUC: [{ci_low:.4f}, {ci_high:.4f}]")

# ── IoD ratio ──
iod_ndup = np.array([d['ecs'] for d in hard_subset if d['pair_type'] == 'near_dup'])
iod_hneg = np.array([d['ecs'] for d in hard_subset if d['pair_type'] == 'hard_neg'])
med_ndup = float(np.median(iod_ndup))
med_hneg = float(np.median(iod_hneg))
iod_ratio = med_ndup / med_hneg if med_hneg > 0 else float('inf')
mw = mannwhitneyu(iod_ndup, iod_hneg, alternative='greater')
mw_p = float(mw.pvalue)
print(f"Median IoD: ndup={med_ndup:.4f} hneg={med_hneg:.4f} ratio={iod_ratio:.3f} p={mw_p:.4e}")

# ── Cohen's d on log-IoD ──
eps = 1e-6
log_ndup = np.log(iod_ndup + eps)
log_hneg = np.log(iod_hneg + eps)
cd = cohens_d(log_ndup, log_hneg)
print(f"Cohen's d on log-IoD: {cd:.4f}")

# ── Length-stratified AUC ──
buckets_map = defaultdict(list)
for d in hard_subset:
    b = length_bucket(d['avg_words'])
    buckets_map[b].append(d)

length_strata_aucs = []
for bkt in ['<200', '200-500', '>500']:
    items = buckets_map[bkt]
    if len(items) < 10:
        length_strata_aucs.append({'bucket': bkt, 'n': len(items), 'auc_jaccard': None, 'auc_combined': None})
        continue
    yb = np.array([d['label'] for d in items])
    if len(np.unique(yb)) < 2:
        length_strata_aucs.append({'bucket': bkt, 'n': len(items), 'auc_jaccard': None, 'auc_combined': None})
        continue
    jb = np.array([d['jaccard'] for d in items])
    eb = np.array([d['ecs'] for d in items])
    cb = jb + 0.3 * eb
    a_j = float(roc_auc_score(yb, jb))
    a_c = float(roc_auc_score(yb, cb))
    print(f"  Bucket {bkt}: n={len(items)} AUC_jac={a_j:.4f} AUC_comb={a_c:.4f}")
    length_strata_aucs.append({'bucket': bkt, 'n': len(items), 'auc_jaccard': a_j, 'auc_combined': a_c})

# ── Confusion matrix at 80% recall ──
conf_jac = confusion_at_recall(y_hard, jac_hard)
conf_comb = confusion_at_recall(y_hard, combined_hard)
prec_gain = conf_comb['precision'] - conf_jac['precision']
print(f"Confusion @80% recall: Jac prec={conf_jac['precision']:.4f}  Comb prec={conf_comb['precision']:.4f}  gain={prec_gain:.4f}")

# ── Verdicts ──
verdict_auc = 'CONFIRMED' if delta_auc >= 0.03 and ci_low > 0 else \
              ('PARTIAL' if delta_auc >= 0.03 else 'DISCONFIRMED')
verdict_ecs_alone = 'CONFIRMED' if auc_ecs_only > 0.65 else 'DISCONFIRMED'
verdict_iod = 'CONFIRMED' if iod_ratio >= 2.0 and mw_p < 0.01 else \
              ('PARTIAL' if iod_ratio >= 1.5 else 'DISCONFIRMED')
confirmed = sum([v == 'CONFIRMED' for v in [verdict_auc, verdict_ecs_alone, verdict_iod]])
partial = sum([v == 'PARTIAL' for v in [verdict_auc, verdict_ecs_alone, verdict_iod]])
if confirmed >= 2:
    verdict_overall = 'CONFIRMED'
elif confirmed + partial >= 2:
    verdict_overall = 'PARTIAL'
else:
    verdict_overall = 'DISCONFIRMED'

print(f"\nVerdicts:")
print(f"  AUC improvement: {verdict_auc}")
print(f"  ECS alone:       {verdict_ecs_alone}")
print(f"  IoD ratio:       {verdict_iod}")
print(f"  Overall:         {verdict_overall}")

## Results Summary & Visualization

The plots below show the key findings:
- IoD distributions for near-duplicates vs hard-negatives
- AUC comparison across scoring methods

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: IoD distributions
ax = axes[0]
ax.hist(iod_ndup, bins=10, alpha=0.6, label=f'near_dup (med={med_ndup:.1f})', color='steelblue')
ax.hist(iod_hneg, bins=10, alpha=0.6, label=f'hard_neg (med={med_hneg:.1f})', color='tomato')
ax.set_xlabel('ECS (IoD of edit gaps)')
ax.set_ylabel('Count')
ax.set_title('IoD Distribution by Pair Type')
ax.legend(fontsize=8)
ax.text(0.5, 0.92, f'ratio={iod_ratio:.2f}, p={mw_p:.2f}', transform=ax.transAxes,
        ha='center', fontsize=8, color='gray')

# Plot 2: AUC bar chart
ax = axes[1]
methods = ['Jaccard', 'ECS only', 'Jac+ECS']
aucs = [auc_jac, auc_ecs_only, auc_combined]
colors = ['steelblue', 'orange', 'green']
bars = ax.bar(methods, aucs, color=colors, alpha=0.8)
ax.axhline(0.65, color='red', linestyle='--', alpha=0.5, label='ECS threshold (0.65)')
ax.set_ylim(0, 1.15)
ax.set_ylabel('AUC')
ax.set_title('AUC by Scoring Method\n(hard_neg subset)')
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9)
ax.legend(fontsize=8)

# Plot 3: Verdict summary table
ax = axes[2]
ax.axis('off')
rows = [
    ['Criterion', 'Value', 'Verdict'],
    ['AUC delta', f'{delta_auc:+.3f}', verdict_auc],
    ['ECS AUC', f'{auc_ecs_only:.3f}', verdict_ecs_alone],
    ['IoD ratio', f'{iod_ratio:.3f}', verdict_iod],
    ['Cohen\'s d', f'{cd:.3f}', '(neg=wrong dir)'],
    ['OVERALL', '', verdict_overall],
]
colors_table = [['#ddd']*3] + [['white']*3]*(len(rows)-2) + [['#ffcccc']*3 if verdict_overall=='DISCONFIRMED' else ['#ccffcc']*3]
tbl = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
               cellColours=colors_table[1:])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.5)
ax.set_title('Verdict Summary', pad=20)

plt.suptitle('ECS vs Jaccard: Near-Duplicate Detection Validation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY FINDING:")
print(f"  Near-dup IoD median: {med_ndup:.2f}")
print(f"  Hard-neg IoD median: {med_hneg:.2f}")
print(f"  Ratio (ndup/hneg):   {iod_ratio:.3f}  (expected >2.0, got {iod_ratio:.3f})")
print(f"  Direction is WRONG: splice creates ONE contiguous edit block (low IoD),")
print(f"  while hard-negatives have many scattered differences (high IoD).")
print(f"\n  Overall verdict: {verdict_overall}")